In [1]:
import pandas as pd
import duckdb
data = [
    # Device A：有完整的上月同日记录
    ["A", "2026-01-15", 10],
    ["A", "2026-02-15", 14],
    ["A", "2026-03-15", 11],

    # Device B：中间缺少 2026-02-10
    ["B", "2026-01-10", 5],
    ["B", "2026-03-10", 9],

    # Device C：有部分上月同日记录
    ["C", "2026-01-20", 20],
    ["C", "2026-02-20", 18],
    ["C", "2026-04-20", 25],

    # Device D：只有后期数据，没有上月同日
    ["D", "2026-03-05", 30],
    ["D", "2026-04-05", 28],

    # Device E：用来提醒不能用 30 天前代替上个月
    ["E", "2026-01-01", 100],
    ["E", "2026-02-01", 120],
    ["E", "2026-03-01", 90],
]

df = pd.DataFrame(
    data,
    columns=["device_id", "stat_date", "alarm_count"]
)

df["stat_date"] = pd.to_datetime(df["stat_date"])

print(df)


   device_id  stat_date  alarm_count
0          A 2026-01-15           10
1          A 2026-02-15           14
2          A 2026-03-15           11
3          B 2026-01-10            5
4          B 2026-03-10            9
5          C 2026-01-20           20
6          C 2026-02-20           18
7          C 2026-04-20           25
8          D 2026-03-05           30
9          D 2026-04-05           28
10         E 2026-01-01          100
11         E 2026-02-01          120
12         E 2026-03-01           90


## 题目要求

### 分别使用 SQL 和 Pandas 完成：

计算每个设备每天的报警次数与上个月同日相比变化了多少。

这里的“上个月同日”指：

同一个 device_id
并且日期是当前日期的上一个自然月同一天

例如：

- 2026-02-15 对比 2026-01-15
- 2026-03-15 对比 2026-02-15
- 2026-04-05 对比 2026-03-05

如果上个月同日没有记录，则：

- `alarm_count_previous_month = NULL / NaN`
- `alarm_count_diff_month = NULL / NaN`

###最终输出字段:

- `device_id`
- `stat_date`
- `alarm_count`
- `alarm_count_previous_month`
- `alarm_count_diff_month`

In [6]:
# SQL轨道

query = """
WITH previous_table AS (
    SELECT
        device_id,
        
        stat_date + INTERVAL 1 MONTH AS stat_date,
        alarm_count AS alarm_count_previous_month
    FROM df
)
SELECT
    curr.device_id,
    curr.stat_date,
    curr.alarm_count,
    prev.alarm_count_previous_month,
    curr.alarm_count - prev.alarm_count_previous_month AS alarm_count_diff_month
FROM df AS curr
LEFT JOIN previous_table AS prev
    ON curr.device_id = prev.device_id
        AND curr.stat_date = prev.stat_date
ORDER BY device_id,stat_date
"""
df_sql = duckdb.execute(query).fetchdf()
df_sql

,device_id,stat_date,alarm_count,alarm_count_previous_month,alarm_count_diff_month
0,A,2026-01-15,10,<NA>,<NA>
1,A,2026-02-15,14,10,4
2,A,2026-03-15,11,14,-3
3,B,2026-01-10,5,<NA>,<NA>
4,B,2026-03-10,9,<NA>,<NA>
5,C,2026-01-20,20,<NA>,<NA>
6,C,2026-02-20,18,20,-2
7,C,2026-04-20,25,<NA>,<NA>
8,D,2026-03-05,30,<NA>,<NA>
9,D,2026-04-05,28,30,-2


In [13]:
# PANDAS轨道

df_prev = (
    df
    .assign(
        stat_date = lambda x:(
            x['stat_date'] + pd.DateOffset(months=1)
        )
    )
    .rename(
        columns = {'alarm_count':'alarm_count_previous_month'}
    )
)
df_pd = (
    df
    .merge(
        df_prev,
        on = ['device_id','stat_date'],
        how = 'left'
    )
    .assign(
        alarm_count_diff_month = lambda x:(
            x['alarm_count'] - x['alarm_count_previous_month']
        )
    )
    .sort_values(by=['device_id','stat_date'])
    .reset_index(drop=True)
)
df_pd

,device_id,stat_date,alarm_count,alarm_count_previous_month,alarm_count_diff_month
0,A,2026-01-15,10,NaN,NaN
1,A,2026-02-15,14,10.0,4.0
2,A,2026-03-15,11,14.0,-3.0
3,B,2026-01-10,5,NaN,NaN
4,B,2026-03-10,9,NaN,NaN
5,C,2026-01-20,20,NaN,NaN
6,C,2026-02-20,18,20.0,-2.0
7,C,2026-04-20,25,NaN,NaN
8,D,2026-03-05,30,NaN,NaN
9,D,2026-04-05,28,30.0,-2.0
